In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [ ]:
# Silver dan o'qish
df_silver = spark.read.format("delta").load(
    "abfss://6c47144b-4924-4537-8fce-73a81f93118b@onelake.dfs.fabric.microsoft.com/9eefcb06-4f43-4487-911e-8ce8c88e442e/Tables/dbo/taxi_silver_clean"
)

print("✅ Silver qatorlar:", df_silver.count())

In [ ]:
from pyspark.sql import functions as F

# FactTaxiDaily - pickup_hour va pickup_month bilan
fact_taxi = df_silver.groupBy(
    "pickup_month",
    "pickup_hour",
    "PULocationID",
    "DOLocationID",
    "payment_type"
).agg(
    F.round(F.count("*"),2).alias("total_trips"),
    F.round(F.sum("total_amount"),2).alias("total_revenue"),
    F.round(F.avg("fare_amount"),2).alias("avg_fare"),
    F.round(F.avg("trip_distance"),2).alias("avg_distance"),
    F.round(F.avg("trip_duration_minutes"),2).alias("avg_duration"),
    F.round(F.sum("tip_amount"),2).alias("total_tips"),
    F.round(F.sum("tolls_amount"),2).alias("total_tolls")
)

print("✅ Fact table qatorlar:", fact_taxi.count())
display(fact_taxi)

In [ ]:
from pyspark.sql import functions as F

fact_taxi_save = fact_taxi.select(
    F.col("pickup_month").cast("int").alias("pickup_month"),
    F.col("pickup_hour").cast("int").alias("pickup_hour"),
    F.col("PULocationID").cast("int").alias("PULocationID"),
    F.col("DOLocationID").cast("int").alias("DOLocationID"),
    F.col("payment_type").cast("int").alias("payment_type"),
    F.col("total_trips").cast("long").alias("total_trips"),
    F.round(F.col("total_revenue"), 2).alias("total_revenue"),
    F.round(F.col("avg_fare"), 2).alias("avg_fare"),
    F.round(F.col("avg_distance"), 2).alias("avg_distance"),
    F.round(F.col("avg_duration"), 2).alias("avg_duration"),
    F.round(F.col("total_tips"), 2).alias("total_tips"),
    F.round(F.col("total_tolls"), 2).alias("total_tolls")
)

fact_taxi_save.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbo.fact_taxi_gold")

print("✅ Gold saqlandi!")